### Setup

In [2]:
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

### DinoV3 

Load encoder ONLY (no MST)

In [3]:
model = torch.hub.load(
    'facebookresearch/dinov3',
    'dinov3_vitb16',
    weights="../dinov3/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth",
    trust_repo=True,
    skip_validation=True
).to(device)

model.eval()

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov3_main


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
    )
  )
  (norm): LayerN

In [4]:
# simulate your slice input
x = torch.randn(1, 3, 224, 224).to(device)

Inspect patch embedding directly

In [ ]:
with torch.no_grad():
    x_patch = model.patch_embed(x)

print("Patch embedding output:", x_patch.shape)

with torch.no_grad():
    out = model.forward_features(x)

print(out.keys())

Patch embedding output: torch.Size([1, 14, 14, 768])


In [16]:
# Patch tokens (without CLS):
patch_tokens = out["x_norm_patchtokens"]
print("Patch tokens:", patch_tokens.shape)

# CLS token:
cls_token = out["x_norm_clstoken"]
print("CLS token:", cls_token.shape)

# storage token
storage_token = out["x_storage_tokens"]
print("Storage token:", storage_token.shape)

#normed_patch_tokens :
normed_patch_tokens = out["x_norm_patchtokens"]
print("Normed patch tokens:", normed_patch_tokens.shape)

#prenorm:
prenorm = out["x_prenorm"]
print("Prenorm output:", prenorm.shape)  

Patch tokens: torch.Size([1, 196, 768])
CLS token: torch.Size([1, 768])
Storage token: torch.Size([1, 4, 768])
Normed patch tokens: torch.Size([1, 196, 768])
Prenorm output: torch.Size([1, 201, 768])


### Test with DinoV2 wih register ViTs

In [21]:
model = torch.hub.load('facebookresearch/dinov2', 
                       f'dinov2_vits14_reg',
                       trust_repo=True, 
                       skip_validation=True

).to(device)

model.eval()

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_reg4_pretrain.pth" to /home/jovyan/.cache/torch/hub/checkpoints/dinov2_vits14_reg4_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:00, 110MB/s] 


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [23]:
# simulate your slice input
x = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    x_patch = model.patch_embed(x)

print("Patch embedding output:", x_patch.shape)

with torch.no_grad():
    out = model.forward_features(x)

print("Forward features output keys:", out.keys())

Patch embedding output: torch.Size([1, 256, 384])
Forward features output keys: dict_keys(['x_norm_clstoken', 'x_norm_regtokens', 'x_norm_patchtokens', 'x_prenorm', 'masks'])


In [25]:
# Patch tokens (without CLS):
patch_tokens = out["x_norm_patchtokens"]
print("Patch tokens:", patch_tokens.shape)

# CLS token:
cls_token = out["x_norm_clstoken"]
print("CLS token:", cls_token.shape)

# register token
register_token = out["x_norm_regtokens"]
print("Register token:", register_token.shape)

#normed_patch_tokens :
normed_patch_tokens = out["x_norm_patchtokens"]
print("Normed patch tokens:", normed_patch_tokens.shape)

#prenorm:
prenorm = out["x_prenorm"]
print("Prenorm output:", prenorm.shape)  

Patch tokens: torch.Size([1, 256, 384])
CLS token: torch.Size([1, 384])
Register token: torch.Size([1, 4, 384])
Normed patch tokens: torch.Size([1, 256, 384])
Prenorm output: torch.Size([1, 261, 384])


### Test with DinoV2 wihout register ViTs

In [26]:
model = torch.hub.load('facebookresearch/dinov2', 
                       f'dinov2_vits14',
                       trust_repo=True, 
                       skip_validation=True

).to(device)

model.eval()

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [27]:
# simulate your slice input
x = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    x_patch = model.patch_embed(x)

print("Patch embedding output:", x_patch.shape)

with torch.no_grad():
    out = model.forward_features(x)

print("Forward features output keys:", out.keys())

Patch embedding output: torch.Size([1, 256, 384])
Forward features output keys: dict_keys(['x_norm_clstoken', 'x_norm_regtokens', 'x_norm_patchtokens', 'x_prenorm', 'masks'])


In [28]:
# Patch tokens (without CLS):
patch_tokens = out["x_norm_patchtokens"]
print("Patch tokens:", patch_tokens.shape)

# CLS token:
cls_token = out["x_norm_clstoken"]
print("CLS token:", cls_token.shape)

# register token
register_token = out["x_norm_regtokens"]
print("Register token:", register_token.shape)

#normed_patch_tokens :
normed_patch_tokens = out["x_norm_patchtokens"]
print("Normed patch tokens:", normed_patch_tokens.shape)

#prenorm:
prenorm = out["x_prenorm"]
print("Prenorm output:", prenorm.shape)  

Patch tokens: torch.Size([1, 256, 384])
CLS token: torch.Size([1, 384])
Register token: torch.Size([1, 0, 384])
Normed patch tokens: torch.Size([1, 256, 384])
Prenorm output: torch.Size([1, 257, 384])


### MST with DinoV3